In [1]:
# 0. Environment Setup

# Clone the gemma repository
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository for UI/UX
!git clone https://github.com/google-deepmind/dialog.git || true
!pip install -q ./dialog

# Ensure local modules are in path
import sys
import os
sys.path.append(os.getcwd())

Cloning into 'kauldron'...
remote: Enumerating objects: 9553, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 9553 (delta 52), reused 42 (delta 30), pack-reused 9384 (from 3)
Receiving objects: 100% (9553/9553), 2.84 MiB | 33.00 MiB/s, done.
Resolving deltas: 100% (6860/6860), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 58.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB

In [3]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 284, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 284 (delta 37), reused 67 (delta 36), pack-reused 214 (from 1)
Receiving objects: 100% (284/284), 24.08 MiB | 28.48 MiB/s, done.
Resolving deltas: 100% (167/167), done.
/content/Titans_jax


In [4]:
import importlib
import jax
import os
import gemma_titans
# importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans, Gemma_Titans_Config
from titans_ckpts import SkipTitans
import titans_tree_utils
from gemma import gm

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"


JAX Backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


In [ ]:
import google.colab
import shutil
import orbax.checkpoint as ocp

if os.path.exists('/content/Titans_jax/saved_titans_delta.zip'):
    shutil.unpack_archive('/content/Titans_jax/saved_titans_delta.zip',"./saved_titans_delta")

def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()
    return checkpointer.restore(load_path)

import jax.numpy as jnp
gemma_config = Gemma_Titans_Config

model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)
print("Loading weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)
loaded_titans_params = load_titans_weights("./saved_titans_delta")
merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)

tokenizer = gm.text.Gemma3Tokenizer()

## 4. Interactive Dialogue with `google-deepmind/dialog`

In [ ]:
import dialog
from gemma import gm
import jax

# Initialize Sampler and Conversation
sampler = gm.text.Sampler(
    model=model,
    params=merged_params,
    tokenizer=tokenizer,
)

conv = dialog.Conversation()
conv.show()

def chat(user_input: str):
    global conv
    # Add user message
    conv += dialog.User(user_input)
    
    # Convert conversation to prompt (Gemma 3 format)
    prompt = conv.as_text(training=False)
    
    # Generate response
    # Note: Sampler handles the caching of Titans memory automatically
    response_text = sampler.sample(prompt, max_new_tokens=128)
    
    # Add model response to UI
    conv += dialog.Model(response_text)
    conv.show()

# Example usage:
# chat("Привет! Кто такие Титаны в мифологии?")